# 60 — DS5 DyS v2: Per-Participant Potential Well V(x) (DS5_02 A1)

**Branch:** DS5_02 Phase A Stage 1, item A1  
**Predecessor:** `42_ds5_dynamical_systems.ipynb` (original, FLAG2 active — median split)  
**New notebook rule:** Notebook 42 is never modified.

## What changes (A1)

DS5_01 FLAG2 (notebook 42): `behaviorally_deterministic = 1 if sig_hat < median(sig_hat)` —
a sample-relative median split with no data-grounded justification.

This notebook replaces it with a **per-participant KDE of the daily X_t series → V(x) = −log p̂(x)
→ fixed-point detection via dV/dx = 0, stability via d²V/dx²**.

## Definition of V(x)

- X_t = daily override rate ∈ [0,1] (same proxy as notebook 43 StoP)
- KDE with reflection at boundaries (bounded domain correction)
- V(x) = −log p̂(x), normalized so min(V) = 0
- Fixed points: locations where dV/dx changes sign
- Stable FP (attractor): d²V/dx² > 0 (potential minimum)
- Unstable FP (repeller): d²V/dx² < 0 (potential maximum)

## Output (feeds notebook 60b)

`participant_dynamics_v2.csv` — per-participant V(x) summary:  
fixed-point location, curvature (V''), density at FP.  
Notebook 60b uses `V_curvature_primary` to derive β_v2 for StM.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.signal import savgol_filter

In [ ]:
ROOT      = Path('..')
OUT_CLEAN = ROOT / 'outputs' / 'cleaned'
OUT_DYS   = ROOT / 'outputs' / 'model_outputs' / 'dynamical_systems'
OUT_FIG   = ROOT / 'outputs' / 'model_outputs' / 'dynamical_systems' / 'figures'
OUT_FIG.mkdir(parents=True, exist_ok=True)

# Same constants as notebook 43 StoP
MIN_DAYS      = 7
VALID_OUTCOMES = {'opened', 'dismissed', 'closed'}

# KDE grid
GRID_N = 200
GRID   = np.linspace(0, 1, GRID_N)

In [ ]:
# ── Load + daily X_t construction (identical to notebook 43) ──────────────
beh = pd.read_csv(OUT_CLEAN / 'ds5_behavioral_clean.csv',
                  parse_dates=['event_timestamp_utc'])
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp_utc'], utc=True)
beh['event_date'] = beh['event_timestamp_utc'].dt.normalize()

pre = pd.read_csv(OUT_CLEAN / 'ds5_pre_survey_clean.csv')
cond_map = pre.set_index('participant_id_canonical')['cond']
beh['study_condition'] = beh['participant_id_canonical'].map(cond_map)

beh_fric = beh[beh['event_outcome_simple'].isin(VALID_OUTCOMES)].copy()
beh_fric['opened_flag'] = (beh_fric['event_outcome_simple'] == 'opened').astype(int)

daily = (
    beh_fric
    .groupby(['participant_id_canonical', 'study_condition', 'event_date'])
    .agg(n_opened=('opened_flag', 'sum'), n_events=('opened_flag', 'count'))
    .reset_index()
)
daily['X_t'] = daily['n_opened'] / daily['n_events']
daily = daily.sort_values(['participant_id_canonical', 'event_date']).reset_index(drop=True)

n_days_pp = daily.groupby('participant_id_canonical')['event_date'].nunique()
eligible  = n_days_pp[n_days_pp >= MIN_DAYS].index
daily_elig = daily[daily['participant_id_canonical'].isin(eligible)].copy()

print(f'Daily records  : {len(daily):,}')
print(f'Eligible (≥{MIN_DAYS} days): {len(eligible)} participants')

In [ ]:
# ── V(x) estimation function ───────────────────────────────────────────────

def estimate_potential_well(x_series, bw_method='silverman',
                            savgol_window=21, savgol_poly=3):
    """
    Estimate V(x) = -log p̂(x) for a bounded [0,1] X_t series.

    Uses reflection-corrected KDE and Savitzky-Golay smoothing.
    Returns fixed-point summary dict.

    low_confidence=True when n_xt < 10 (KDE unreliable with very few points).
    """
    x = np.asarray(x_series, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)

    base = dict(
        n_xt=n, kde_bw=np.nan,
        has_stable_fp=False, n_fps=0, n_stable_fps=0,
        x_fp_primary=np.nan, V_curvature_primary=np.nan, p_at_fp_primary=np.nan,
        low_confidence=(n < 10),
    )

    if n < 5:
        return base

    # Reflection at [0, 1] for bounded domain KDE
    x_refl = np.concatenate([-x, x, 2 - x])
    try:
        kde = gaussian_kde(x_refl, bw_method=bw_method)
    except Exception:
        return base

    # Evaluate on [0,1] grid; factor-of-3 correction for reflection tripling
    density = kde(GRID) * 3
    density = np.maximum(density, 1e-10)
    dg = GRID[1] - GRID[0]
    density = density / (density.sum() * dg)  # normalize to integrate to 1

    bw_eff = float(kde.factor * np.std(x))

    # V(x) = -log p̂(x), shifted so min = 0
    V = -np.log(density)
    V = V - V.min()

    # Smooth V via Savitzky-Golay to stabilize numerical derivatives
    wlen = savgol_window
    wlen = min(wlen, GRID_N // 4 * 2 - 1)
    if wlen < 5: wlen = 5
    if wlen % 2 == 0: wlen -= 1
    try:
        V_sm = savgol_filter(V, window_length=wlen, polyorder=savgol_poly)
    except Exception:
        V_sm = V

    dV  = np.gradient(V_sm, GRID)
    d2V = np.gradient(dV,   GRID)

    # Fixed points: sign changes in dV, interior only (avoid boundary artefacts)
    margin = 5  # skip first/last 5 grid points
    dV_interior = dV[margin:-margin]
    sc_idx = np.where(np.diff(np.sign(dV_interior)) != 0)[0] + margin

    fps = []
    for i in sc_idx:
        x_fp   = float((GRID[i] + GRID[i+1]) / 2)
        curv   = float((d2V[i] + d2V[i+1]) / 2)
        p_fp   = float((density[i] + density[i+1]) / 2)
        stable = curv > 0
        fps.append({'x': x_fp, 'curvature': curv, 'p': p_fp, 'stable': stable})

    stable_fps = [fp for fp in fps if fp['stable']]
    n_fps      = len(fps)
    n_stable   = len(stable_fps)

    if n_fps == 0:
        return {**base, 'n_xt': n, 'kde_bw': bw_eff}

    # Primary = highest-density stable FP; fall back to any FP if no stable found
    pool    = stable_fps if stable_fps else fps
    primary = max(pool, key=lambda f: f['p'])

    return dict(
        n_xt=n, kde_bw=bw_eff,
        has_stable_fp=(n_stable > 0),
        n_fps=n_fps, n_stable_fps=n_stable,
        x_fp_primary=primary['x'],
        V_curvature_primary=primary['curvature'],
        p_at_fp_primary=primary['p'],
        low_confidence=(n < 10),
    )

print('estimate_potential_well() defined.')

In [ ]:
# ── Per-participant V(x) computation ──────────────────────────────────────
records = []
for pid, grp in daily_elig.groupby('participant_id_canonical'):
    cond     = grp['study_condition'].iloc[0]
    x_series = grp.sort_values('event_date')['X_t'].values
    res = estimate_potential_well(x_series)
    res['participant_id_canonical'] = pid
    res['study_condition']          = cond
    records.append(res)

dys_v2 = pd.DataFrame(records)
print(f'Computed: {len(dys_v2)} participants')
print(f'Has stable FP   : {dys_v2["has_stable_fp"].sum()} ({dys_v2["has_stable_fp"].mean()*100:.1f}%)')
print(f'No stable FP    : {(~dys_v2["has_stable_fp"]).sum()}')
print(f'Low confidence  : {dys_v2["low_confidence"].sum()} (n_xt < 10)')
print(f'\nFP count distribution:')
print(dys_v2['n_fps'].value_counts().sort_index())

In [ ]:
# ── Compare v2 with original DyS behaviorally_deterministic ───────────────
orig = pd.read_csv(OUT_DYS / 'participant_dynamics_parameters.csv',
                   usecols=['participant_id_canonical', 'study_condition',
                            'behaviorally_deterministic', 'is_dys_eligible',
                            'sig_hat', 'x_mean'])

compare = orig.merge(
    dys_v2[['participant_id_canonical', 'has_stable_fp', 'low_confidence',
            'n_fps', 'n_stable_fps', 'x_fp_primary', 'V_curvature_primary',
            'p_at_fp_primary']],
    on='participant_id_canonical', how='left'
)

# DyS_v2 eligibility: same as original (has_stable_fp and ≥7 days)
compare['is_dys_eligible_v2'] = compare['has_stable_fp'].fillna(False)

print('=== COMPARISON: original vs v2 determinism classification ===')
print(f'\nOriginal behaviorally_deterministic (median-split FLAG2):')
print(f'  True (det.)   : {compare["behaviorally_deterministic"].sum()}')
print(f'  False (noise) : {(compare["behaviorally_deterministic"]==0).sum()}')
print(f'\nv2 has_stable_fp (KDE potential well):')
print(f'  Has stable FP : {compare["is_dys_eligible_v2"].sum()}')
print(f'  No stable FP  : {(~compare["is_dys_eligible_v2"]).sum()}')

# Cross-tabulation
print(f'\nCross-tab (original det. vs v2 stable FP):')
ct = pd.crosstab(
    compare['behaviorally_deterministic'].astype(bool).map({True:'orig_det', False:'orig_noise'}),
    compare['is_dys_eligible_v2'].astype(bool).map({True:'v2_stable', False:'v2_no_fp'})
)
assert ct.size > 0, 'crosstab is empty — check dtypes of the mapped columns'
print(ct)

In [ ]:
# ── Condition-level summary ───────────────────────────────────────────────
print('Has stable FP by condition (v2):')
print(compare.groupby('study_condition')['is_dys_eligible_v2']
             .agg(['sum','count'])
             .assign(pct=lambda x: (x['sum']/x['count']*100).round(1)))
print()
print('Original behaviorally_deterministic by condition:')
print(compare.groupby('study_condition')['behaviorally_deterministic']
             .agg(['sum','count'])
             .assign(pct=lambda x: (x['sum']/x['count']*100).round(1)))

In [ ]:
# ── Fixed-point location distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
elig_fp = compare.dropna(subset=['x_fp_primary'])
for cond, color in [('control','#4dac26'), ('default','#d01c8b'), ('reintervention','#f1b6da')]:
    vals = elig_fp.loc[elig_fp['study_condition']==cond, 'x_fp_primary']
    ax.hist(vals, bins=20, alpha=0.5, color=color, label=f'{cond} (n={len(vals)})', density=True)
ax.set_xlabel('x_fp_primary  (stable FP location)')
ax.set_ylabel('Density')
ax.set_title('Stable FP location by condition (v2)')
ax.legend(fontsize=8)

ax = axes[1]
elig_fp2 = compare.dropna(subset=['V_curvature_primary'])
ax.hist(elig_fp2['V_curvature_primary'], bins=30, color='#666', alpha=0.7)
ax.set_xlabel('V_curvature_primary  (V\'\'(x*) at stable FP)')
ax.set_ylabel('Count')
ax.set_title('Potential-well curvature distribution')
ax.axvline(0, color='red', lw=1.5, ls='--', label='V\'\'=0 boundary')
ax.legend(fontsize=8)

fig.suptitle('DS5 DyS v2 — Potential-Well Fixed-Point Summary', fontsize=12)
plt.tight_layout()
fig.savefig(OUT_FIG / 'fig_dys_v2_fp_distributions.png', bbox_inches='tight', dpi=200)
print('Saved: fig_dys_v2_fp_distributions.png')
plt.show()

In [ ]:
# ── Example V(x) profiles (illustrative) ──────────────────────────────────
# Select 6 participants: 2 with clear stable FP, 2 with no FP, 2 low-confidence

def compute_vx_for_plot(x_series):
    """Return (GRID, density, V_smooth) for plotting."""
    x = np.asarray(x_series, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 5:
        return GRID, np.full(GRID_N, np.nan), np.full(GRID_N, np.nan)
    x_refl  = np.concatenate([-x, x, 2 - x])
    kde     = gaussian_kde(x_refl, bw_method='silverman')
    density = np.maximum(kde(GRID) * 3, 1e-10)
    dg      = GRID[1] - GRID[0]
    density = density / (density.sum() * dg)
    V       = -np.log(density); V -= V.min()
    wlen    = 21 if GRID_N > 42 else 9
    if wlen % 2 == 0: wlen -= 1
    V_sm    = savgol_filter(V, window_length=wlen, polyorder=3)
    return GRID, density, V_sm

has_fp   = compare[compare['is_dys_eligible_v2']]['participant_id_canonical'].iloc[:2].tolist()
no_fp    = compare[~compare['is_dys_eligible_v2'] & compare['has_stable_fp'].notna()]['participant_id_canonical'].iloc[:2].tolist()
low_conf = compare[compare['low_confidence']==True]['participant_id_canonical'].iloc[:2].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
examples  = has_fp + no_fp + low_conf
labels    = ['stable FP']*2 + ['no stable FP']*2 + ['low confidence']*2

for ax, pid, lbl in zip(axes.flat, examples, labels):
    x_s   = daily_elig[daily_elig['participant_id_canonical']==pid]['X_t'].values
    g, d, v = compute_vx_for_plot(x_s)
    fp_row = compare[compare['participant_id_canonical']==pid]
    ax2   = ax.twinx()
    ax.plot(g, d, color='#2196F3', lw=1.5, alpha=0.8, label='p̂(x)')
    ax2.plot(g, v, color='#FF5722', lw=1.5, alpha=0.8, label='V(x)', ls='--')
    if not fp_row['x_fp_primary'].isna().values[0]:
        xfp = fp_row['x_fp_primary'].values[0]
        ax2.axvline(xfp, color='#9C27B0', lw=1.5, ls=':', label=f'x*={xfp:.2f}')
    ax.set_xlabel('X_t (override rate)')
    ax.set_ylabel('p̂(x)', color='#2196F3')
    ax2.set_ylabel('V(x)', color='#FF5722')
    ax.set_title(f'[{lbl}]\nn={len(x_s)} days')
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labs1+labs2, fontsize=7, loc='upper right')

fig.suptitle('Example V(x) Profiles — DS5 DyS v2', fontsize=12)
plt.tight_layout()
fig.savefig(OUT_FIG / 'fig_dys_v2_example_profiles.png', bbox_inches='tight', dpi=200)
print('Saved: fig_dys_v2_example_profiles.png')
plt.show()

In [ ]:
# ── Save output ────────────────────────────────────────────────────────────
out_cols = [
    'participant_id_canonical', 'study_condition', 'n_xt', 'kde_bw',
    'has_stable_fp', 'n_fps', 'n_stable_fps',
    'x_fp_primary', 'V_curvature_primary', 'p_at_fp_primary',
    'low_confidence',
]
out_path = OUT_DYS / 'participant_dynamics_v2.csv'
dys_v2[out_cols].to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Rows: {len(dys_v2)}, Columns: {out_cols}')

In [ ]:
# ── A1 summary for DS5_02 review gate ─────────────────────────────────────
n_orig_det = int(compare['behaviorally_deterministic'].sum())
n_v2_stable = int(compare['is_dys_eligible_v2'].sum())
n_total     = len(compare)

print('=== DS5_02 A1 VERDICT ===')
print(f'Original FLAG2 behaviorally_deterministic: {n_orig_det}/{n_total} ({100*n_orig_det/n_total:.1f}%)')
print(f'v2 has_stable_fp (KDE potential well):     {n_v2_stable}/{n_total} ({100*n_v2_stable/n_total:.1f}%)')
print()

# Agreement between methods — full-sample concordance over ALL participants.
# (V_curvature_primary is non-null exactly when has_stable_fp is True, so
#  restricting to it would give the v2 positive predictive value, not agreement.)
agree = (
    compare['behaviorally_deterministic'].astype(bool) == compare['is_dys_eligible_v2'].astype(bool)
).mean()
ppv = (
    compare.loc[compare['is_dys_eligible_v2'].astype(bool), 'behaviorally_deterministic'].astype(bool)
).mean()
print(f'Full-sample agreement (all {len(compare)} participants): {agree*100:.1f}%')
print(f'(v2-positive subgroup also orig-deterministic, i.e. PPV: {ppv*100:.1f}%)')
print()
print('Next: notebook 60b uses V_curvature_primary to derive β_v2 for StM (A2).')

## Summary

**Outputs:**
- `outputs/model_outputs/dynamical_systems/participant_dynamics_v2.csv`
- `outputs/model_outputs/dynamical_systems/figures/fig_dys_v2_fp_distributions.png`
- `outputs/model_outputs/dynamical_systems/figures/fig_dys_v2_example_profiles.png`

**Key column for notebook 60b:** `V_curvature_primary` = V''(x*) at stable FP  
→ used as β_v2 (curvature replaces parabolic approximation from FLAG A)

**Review gate input A1:** count and set of participants with stable FP vs original median-split.